# Census Analysis

## Imports and Variables

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
CENSUS_PATH = Path("datasets/census.csv")

## Dataset

In [ ]:
census_df = pd.read_csv(CENSUS_PATH)
display(census_df.sample(5))

## Data Visualization

In [ ]:
sns.countplot(x=census_df['income'])

In [ ]:
sns.histplot(
    data=census_df,
    x='age',
    bins=10,
    kde=True
)

In [ ]:
sns.histplot(
    data=census_df,
    x='education-num',
    bins=8,
    kde=True
)

In [ ]:
sns.histplot(
    data=census_df,
    x='hour-per-week',
    bins=10,
    kde=True
)

In [ ]:
px.treemap(
    data_frame=census_df,
    path=[
        'workclass',
        'age'
    ]
).show()

In [ ]:
px.parallel_categories(
    data_frame=census_df,
    dimensions=[
        'education-num',
        'income'
    ]
).show()

## Feature and Target Split

In [ ]:
X_census = census_df.iloc[:, 0:14].values
y_census = census_df.iloc[:, 14].values

print(X_census)
print(y_census)

## Categorial Attributes Treatment

### Auxilar consts

In [ ]:
categorical_columns = census_df.select_dtypes(include=['str', 'object', 'category']).columns

categorical_indices = [
    census_df.columns.get_loc(col)
    for col in categorical_columns
    if col != 'income'
]

print(categorical_indices)

### Label Encoder

In [ ]:
for i in categorical_indices:
    X_census[:, i] = LabelEncoder().fit_transform(X_census[:, i])

print(X_census)
print(X_census.shape)

### One Hot Encoder

In [ ]:
ct = ColumnTransformer(
    transformers=[(
        'onehot', 
        OneHotEncoder(), 
        categorical_indices
    )],
    remainder='passthrough'
)
X_census = ct.fit_transform(X_census).toarray()

print(X_census)
print(X_census.shape)

### One-Hot Encoding of Categorical Features

The `OneHotEncoder` can be applied directly to categorical columns without using `LabelEncoder` first. This approach preserves the original categorical values and allows the encoder to generate meaningful feature names, such as `gender_M` or `education_High`, using `get_feature_names_out()`. This is preferable to manually applying `LabelEncoder` to each categorical column before one-hot encoding.

```python
from sklearn.preprocessing import OneHotEncoder

categorical_columns = ['gender', 'education']

encoder = OneHotEncoder(sparse_output=False)

X_encoded = encoder.fit_transform(
    census_df[categorical_columns]
)

feature_names = encoder.get_feature_names_out(categorical_columns)

encoded_df = pd.DataFrame(
    X_encoded,
    columns=feature_names,
    index=census_df.index
)

encoded_df.head()
```

This produces columns such as `gender_F`, `gender_M`, `education_College`, and `education_High`.